In [1]:
import os
import torch
from llava.model.builder import load_pretrained_model

def main(src="microsoft/llava-rad",
         base="lmsys/vicuna-7b-v1.5",
         out_dir="./checkpoints/llavarad-merged",
         projector_path="./checkpoints/llavarad_merged/mm_projector.bin",
         dtype="fp16",
         device="cuda"):
    os.makedirs(out_dir, exist_ok=True)
    torch_dtype = torch.float16 if dtype == "fp16" else torch.bfloat16

    # Load exactly like eval: this merges LoRA into base and returns a plain LLaVA model
    tokenizer, model, _, _ = load_pretrained_model(
        model_path=src,
        model_base=base,
        model_name="llavarad",   # triggers the LoRA+merge path in your builder
        load_8bit=False,
        load_4bit=False,
        device_map="auto",
        device=device,
    )
    # Save merged base (optional, handy for training without --pretrain_mm_mlp_adapter)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f"Saved merged model to: {out_dir}")

    # Export projector weights in the expected format
    state = {}
    mm_proj = model.get_model().mm_projector.state_dict()
    for k, v in mm_proj.items():
        state[f"mm_projector.{k}"] = v.to(dtype=torch_dtype, device="cpu")

    # Optional: include embed_tokens for the mm_use_im_start_end path
    try:
        state["model.embed_tokens.weight"] = (
            model.get_input_embeddings().weight.detach().to(dtype=torch_dtype, device="cpu")
        )
        print("Included model.embed_tokens.weight in projector file.")
    except Exception:
        print("Skipped embed_tokens export (not required unless mm_use_im_start_end=True).")

    torch.save(state, projector_path)
    print(f"Exported projector to: {projector_path}")

if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="microsoft/llava-rad")
    ap.add_argument("--base", default="lmsys/vicuna-7b-v1.5")
    ap.add_argument("--out_dir", default="./checkpoints/llavarad-merged")
    ap.add_argument("--projector_path", default="./checkpoints/llavarad_merged/mm_projector.bin")
    ap.add_argument("--dtype", default="fp16", choices=["fp16", "bf16"])
    ap.add_argument("--device", default="cuda")
    args = ap.parse_args()
    main(args.src, args.base, args.out_dir, args.projector_path, args.dtype, args.device)

/home/csgrad/mbhosale/miniconda3/envs/llavarad/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/csgrad/mbhosale/miniconda3/envs/llavarad/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/csgrad/mbhosale/miniconda3/envs/llavarad/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c106detail23torchInternalAssertFailEPKcS2_jS2_RKSs'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
usage: ipykernel_launcher.py [-h] [--src SRC] [--base BASE]
                             [--out_dir OUT_DIR]
                      

SystemExit: 2

/home/csgrad/mbhosale/.local/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3468: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
from datasets import Dataset
import os
cnt = 0
path = "/a2il/data/mbhosale/MrFair/Padchest/rrg24-shared-task-bionlp/StanfordAIMI___rrg24-shared-task-bionlp/default/0.0.0/e0aece58181fdcd02033180057261846209019c4/"
for f in os.listdir(os.path.dirname(path)):
    if not f.endswith(".arrow"):
        continue
    ds = Dataset.from_file(os.path.join(path, f))
    cnt += len(ds)


In [25]:
ds = Dataset.from_file("/a2il/data/mbhosale/MrFair/Padchest/rrg24-shared-task-bionlp/StanfordAIMI___rrg24-shared-task-bionlp/default/0.0.0/e0aece58181fdcd02033180057261846209019c4/rrg24-shared-task-bionlp-train-00010-of-00143.arrow")

In [26]:
ds[100]

{'source': 'PadChest',
 'images_path': ['data/padchest//images/216840111366964013515091760022012310111427454_01-157-053.png'],
 'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x566>],
 'impression': '',
 'findings': 'Mild right convex dorsal scoliosis. The rest without significant findings.'}

In [30]:
ds[120]['images'][0].size

(624, 512)